Deep Neural Network Architectures, such as RNNs, LSTMs, GRUs, and Transformers, are used to build deep learning models for Natural Language Processing (NLP) problems.

Building Deep Learning Models for NLP

Deep learning is ideal for NLP problems that involve complex patterns, high-dimensional data, or sequential dependencies that traditional methods struggle to capture. Architectures like RNNs, LSTMs, GRUs, and Transformers excel in handling such tasks, especially when large datasets and rich contextual understanding are required.

So, let’s get started with building a deep learning model for NLP by importing a dataset. We’ll use the text of a popular book “The Adventures of Sherlock Holmes” as our dataset.Here’s how to load the data:

Upload data

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving sherlock-holm.es_stories_plain-text_advs.txt to sherlock-holm.es_stories_plain-text_advs (2).txt


In [ ]:
with open("sherlock-holm.es_stories_plain-text_advs.txt", "r") as file:
    text = file.read()

Step 1: Preprocessing the Text

Text preprocessing is essential for deep learning models to perform well. Here’s how we clean the text:

1. Remove special characters and punctuation.
2. Convert the text to lowercase.
3. Replace multiple spaces with a single space.

Here’s how to implement these text preprocessing steps:









In [ ]:
import re

def preprocess_text(text):
    text = re.sub(r'\s+', ' ', text)  # replace multiple spaces with a single space
    text = re.sub(r'[^\w\s]', '', text)  # remove punctuation
    text = text.lower()
    return text

text = preprocess_text(text)

Step 2: Tokenization and Sequence Preparation

Next, we need to tokenize the text (convert words to numerical indices) and prepare sequences for training the model. Using Keras’ Tokenizer, we can create a vocabulary of words from the text:

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])
vocab_size = len(tokenizer.word_index) + 1  # add 1 for padding token

For tasks like next word prediction, we create sequences where:

* The first n words are the input.
* The n+1 word is the output.

Here’s how to create such sequences:

In [ ]:
from keras.preprocessing.sequence import pad_sequences

sequence_length = 5  # length of input sequences
sequences = []

# convert text into numerical sequences
words = tokenizer.texts_to_sequences([text])[0]
for i in range(sequence_length, len(words)):
    sequences.append(words[i - sequence_length:i + 1])

# split into inputs (X) and outputs (y)
sequences = pad_sequences(sequences, maxlen=sequence_length + 1, padding='pre')
X, y = sequences[:, :-1], sequences[:, -1]

Step 3: Building the Deep Learning Model

We’ll use an Embedding layer to represent words as dense vectors, followed by an LSTM layer for sequence learning, and a Dense layer for output predictions:

In [ ]:
from keras.models import Sequential
from keras.layers import Embedding, LSTM, Dense

model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=100, input_length=sequence_length),
    LSTM(128, return_sequences=False),  # LSTM processes sequences
    Dense(vocab_size, activation='softmax')  # predict next word
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Next, train the model using the prepared input-output pairs:

In [ ]:
model.fit(X, y, epochs=20, batch_size=64)

Epoch 1/20
1634/1634 ━━━━━━━━━━━━━━━━━━━━ 90s 53ms/step - accuracy: 0.0561 - loss: 6.7823
Epoch 2/20
1634/1634 ━━━━━━━━━━━━━━━━━━━━ 140s 52ms/step - accuracy: 0.0923 - loss: 5.8498
Epoch 3/20
1634/1634 ━━━━━━━━━━━━━━━━━━━━ 142s 52ms/step - accuracy: 0.1282 - loss: 5.4009
Epoch 4/20
1634/1634 ━━━━━━━━━━━━━━━━━━━━ 88s 54ms/step - accuracy: 0.1483 - loss: 5.0729
Epoch 5/20
1634/1634 ━━━━━━━━━━━━━━━━━━━━ 166s 69ms/step - accuracy: 0.1630 - loss: 4.7836
Epoch 6/20
1634/1634 ━━━━━━━━━━━━━━━━━━━━ 118s 72ms/step - accuracy: 0.1824 - loss: 4.5154
Epoch 7/20
1634/1634 ━━━━━━━━━━━━━━━━━━━━ 141s 71ms/step - accuracy: 0.1970 - loss: 4.2820
Epoch 8/20
1634/1634 ━━━━━━━━━━━━━━━━━━━━ 114s 70ms/step - accuracy: 0.2222 - loss: 4.0470
Epoch 9/20
1634/1634 ━━━━━━━━━━━━━━━━━━━━ 117s 72ms/step - accuracy: 0.2454 - loss: 3.8369
Epoch 10/20
1634/1634 ━━━━━━━━━━━━━━━━━━━━ 118s 72ms/step - accuracy: 0.2748 - loss: 3.6247
Epoch 11/20
1634/1634 ━━━━━━━━━━━━━━━━━━━━ 143s 73ms/step - accuracy: 0.3019 - loss: 3.4158

Step 4: Using the Model for Next Word Prediction

After training, we can use this deep learning model to predict the next word given a sequence of words. Here’s how to use the model to predict the next word in a sequence:

In [ ]:
import numpy as np

def predict_next_word(seed_text):
    sequence = tokenizer.texts_to_sequences([seed_text])[0]
    sequence = pad_sequences([sequence], maxlen=sequence_length, padding='pre')
    prediction = model.predict(sequence)
    return tokenizer.index_word[np.argmax(prediction)]

print(predict_next_word("Have you ever"))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
heard


Step 5: Using the Model for Text Generation
We can extend the prediction to generate a sequence of words iteratively. The function below generates a sequence of n words based on an initial seed text:

In [ ]:
def generate_text(seed_text, num_words):
    for _ in range(num_words):
        next_word = predict_next_word(seed_text)
        seed_text += " " + next_word
    return seed_text

print(generate_text("Have you ever", 20))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
Have you ever heard of that he knew that he was not a very quickwitted youth though comely to look at and i


We’ve built a deep learning model capable of next word prediction and text generation using Keras. You can extend this foundational model for more advanced NLP tasks like summarization or sentiment analysis by modifying the architecture or training data.

Summary

Deep Neural Network Architectures, such as RNNs, LSTMs, GRUs, and Transformers, are used to build deep learning models for Natural Language Processing (NLP) problems.